# 1b. Re-score the label set with a revised rubric (E1 iteration)

**Settings (right sidebar), before running:**
1. Accelerator: **GPU T4 x2** (not P100: this pipeline refuses it).
2. Internet: **on** (needs a phone-verified account).
3. Add Input:
   - **tutorbias-data**

Run with **Save Version > Save & Run All (Commit)** so it survives closing the tab. Expected GPU time: **~15 min**.
Every step is resumable: if the run stops early, attach this notebook's own output as an extra Input
and run it again; `restore` picks up the partial files and each step continues where it stopped.

Only needed if the judge missed the E1 gate on dev. Scores the 300 label-set replies with rubric v2 (no rationales). Output: `results/judge/label_set_scores_v2.jsonl`.

In [ ]:
# Secrets (sidebar: Add-ons > Secrets). HF_TOKEN is optional (the models are public, but a token avoids
# download rate limits); WANDB_API_KEY is optional (without it, metrics are only written to files).
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for k in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            os.environ[k] = s.get_secret(k)
            print(k, "loaded")
        except Exception:
            print(k, "not set (fine)")
except ImportError:
    print("not on Kaggle")
# Big files go to /tmp (the 20 GB /kaggle/working is saved as the notebook's output; keep it small).
os.environ.update(HF_HOME="/tmp/hf", VENV_ROOT="/tmp", UV_CACHE_DIR="/tmp/uv", PYTHONUNBUFFERED="1")

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d xai_proj2 ] || git clone --depth 1 https://github.com/anvitasuresh/xai_proj2
cd xai_proj2 && git pull --ff-only && git log --oneline -1
nvidia-smi --query-gpu=name,memory.total --format=csv
# Copy data/ and results/ from every attached Input (your data dataset, earlier notebooks' outputs).
PYTHONPATH=src python3 -m tutorbias.kaggle restore --repo .
PYTHONPATH=src python3 -m tutorbias.kaggle check
bash scripts/setup_local.sh gpu > /tmp/setup.log 2>&1 || (tail -30 /tmp/setup.log; exit 1)
echo "GPU env ready"

In [ ]:
%%bash
cd /kaggle/working/xai_proj2
RUBRIC=v2 EXTRA= OUTPUT=results/judge/label_set_scores_v2.jsonl bash scripts/run_stage.sh judge

In [ ]:
%%bash
# Keep the output small and self-contained: drop the venv symlinks (they point into /tmp).
cd /kaggle/working/xai_proj2 && rm -f .venv-gpu .venv-data
du -sh data results 2>/dev/null || true